In [1]:
# Euchre Calculator -- one hand from deal to score, with a loner in it.
import random

import bidding as b
import game
import rotation as r
from fast_search import definitive_winner

In [2]:
# A deal, laid out on purpose rather than shuffled, because the point here is a
# hand that is obviously worth playing alone: seat 0 holds both bowers, the ace
# and king of trump, and an outside ace. Hearts is turned up, so seat 0 can order
# exactly that in round one.
#
# The dealer bids LAST, so with seat 3 dealing the eldest hand is seat 0.
cards = r.parse_hand(
    "JH JD AH KH AS   "      # seat 0 -- both bowers (hearts called makes JD one)
    "AC KC QC AD KD   "      # seat 1
    "TH 9H KS QS JS   "      # seat 2
    "TC 9C JC QD TD   "      # seat 3, the dealer
    "QH   "                  # the up-card
    "TS 9S 9D")              # buried under it
deal = game.deal_from_order(cards, dealer=3)

print(deal.describe())
print("\nbidding order:", deal.bidding_order(), "(eldest first, dealer last)")

  seat 0: JH JD AH KH AS
  seat 1: AC KC QC AD KD
  seat 2: TH 9H KS QS JS
  seat 3 (dealer): TC 9C JC QD TD
  up-card: QH
  buried:  TS 9S 9D

bidding order: [0, 1, 2, 3] (eldest first, dealer last)


In [3]:
# The auction, solved in God Mode: every seat sees all four hands
# and bids the best call available to it, knowing how the rest of the auction and
# the play will go. That is a correct answer to "was this hand worth calling",
# not a model of a real table -- nobody at a table knows any of that.
#
# The eldest hand's three options, on its own team's scale. Bigger is better:
print("seat %d's options:" % deal.first_bidder)
for option, value in b.first_bid_options(deal).items():
    print("  %-12s %+d" % (option, value))

# Loners are opt-in, because in God Mode they change the result on
# only ~1% of deals. On a hand like this one they plainly do.
four_handed = b.solve_bidding(deal)
with_loners = b.solve_bidding(deal, allow_loners=True)

print("\nfour-handed only:", four_handed)
print("loners allowed  :", with_loners)

outcome = with_loners
contract = outcome.contract

print("\nthe auction, bid by bid:")
for step in outcome.line:
    print("  ", step)

print("\ncaller        : seat %d (team %d)" % (contract.caller, contract.caller_team))
print("trump         :", r.suit_name(contract.trump))
print("alone         : seat %d sits out" % contract.sitting)
print("dealer pitched:", r.card_name(contract.discard))
print("\nscore to the calling team: %+d" % contract.solve())
print("net to team 0            : %+d" % outcome.value)

# Both calls take all five tricks, so alone is worth 4 where four-handed is 2.
# Two details the numbers above show:
#
#   * ordering up and passing tie at +2 four-handed, and ties resolve to passing.
#     That is why the four-handed auction runs on into round two and seat 0 ends
#     up naming diamonds for the same +2 it could have had in round one.
#   * sitting seat 2 down takes its TH and 9H out of the game too, so the only
#     trump left against seat 0 is the queen the dealer has just picked up.

seat 0's options:
  pass         +2
  order        +2
  order alone  +4

four-handed only: seat 0 named diamonds -> +2 to team 0
loners allowed  : seat 0 ordered up hearts alone (seat 2 sits out) (dealer pitched TC) -> +4 to team 0

the auction, bid by bid:
   seat 0 orders up hearts alone

caller        : seat 0 (team 0)
trump         : hearts
alone         : seat 2 sits out
dealer pitched: TC

score to the calling team: +4
net to team 0            : +4


In [4]:
# The play. deal_to_engine rotates the called suit into the solver's canonical
# spades-are-trump frame, so cards print as 2-D vectors rather than names -- the
# right bower is [0, 140], the left [0, 135], and [-14, 0] is the ace of spades.
#
# alone=True is what makes this a loner: tricks come out three cards wide, seat 2
# never plays a card, and taking all five pays 4 instead of 2.
score = definitive_winner(
    dealt_hands=r.deal_to_engine(contract.deal.hands, contract.trump),
    starting_player=contract.deal.first_bidder,
    caller=contract.caller,
    alone=contract.alone,
    verbose=True)

print("\nscore to the calling team: %+d" % score)

# The line is what a player would find at the table: the right bower first, then
# the outside ace while nobody has a trump left to steal it, then the rest of the
# trump. Five tricks, alone, +4.

Starting hands:
 [[[  0 140]
  [  0 135]
  [  0 130]
  [  0 120]
  [-14   0]]

 [[ 14   0]
  [ 13   0]
  [ 12   0]
  [  0 -14]
  [  0 -13]]

 [[  0 100]
  [  0  90]
  [-13   0]
  [-12   0]
  [-11   0]]

 [[  9   0]
  [ 11   0]
  [  0 -12]
  [  0 -10]
  [  0 110]]]
Seat 0 called alone; seat 2 sits out.
Trick 1: [[0, 140], [14, 0], [0, 110]]  (played by [0, 1, 3])
Trick 1 winner: 0
Trick 2: [[-14, 0], [0, -13], [9, 0]]  (played by [0, 1, 3])
Trick 2 winner: 0
Trick 3: [[0, 120], [0, -14], [0, -10]]  (played by [0, 1, 3])
Trick 3 winner: 0
Trick 4: [[0, 130], [12, 0], [0, -12]]  (played by [0, 1, 3])
Trick 4 winner: 0
Trick 5: [[0, 135], [13, 0], [11, 0]]  (played by [0, 1, 3])
Trick 5 winner: 0
Final result: [0, 0, 0, 0, 0]

score to the calling team: +4
